In [1]:
%%writefile submission.py

"""
Orbit Wars — Agent v4 (target 750+ ELO)
========================================

Physics overhaul
  • Type-dispatcher: static / orbiting / comet prediction all handled separately
  • 8-iteration intercept solver; convergence tolerance 0.2 (was 4 iter / 0.5)
  • Comet near-sun safety: reject intercept destinations within 22 u of centre
  • 7-candidate sun-bypass routing per side (was 3)

Strategy overhaul
  • Removed taken_targets lock — multi-source coordination via committed_to
  • Target-first greedy: score ALL targets globally; nearest sources fill first
  • Net-ROI scoring: production × productive_turns / (needed × time_discount)
  • Race-condition bonus: boost priority when enemy fleet is already en route
  • Counter-attack: from_planet_id lets us detect freshly weakened planets
  • Strategic-enemy bonus: destroy their highest-production planet first
  • 4-player vs 2-player adaptation (neutral preference, aggression scaling)
  • No hard distance cap — far high-production planets are correctly valued
  • Minimum fleet size (12 ships) to ensure viable travel speed
  • Adaptive garrison: production buffer shrinks in late game
  • Comet ROI gate: require ≥18 productive turns before attempting capture

Turn structure
  Phase 0  Parse + classify
  Phase 1  Emergency reinforcement of threatened planets
  Phase 2  Target-first multi-source coordinated attacks
  Phase 3  Secondary pass (spend leftover ships on partial coverage)
  Phase 4  Frontline consolidation  (rear → front planet)
  Phase 5  Endgame all-in  (last 35 turns)
"""

import math
from kaggle_environments.envs.orbit_wars.orbit_wars import Planet, Fleet

# ─── Engine constants ─────────────────────────────────────────────────────────
CENTER_X = CENTER_Y  = 50.0
SUN_R           = 10.0      # true sun collision radius
SUN_SAFETY      = 1.5       # extra clearance added for path checks
MAX_SPEED       = 6.0       # maximum fleet speed (units / turn)
ROTATION_LIMIT  = 50.0      # orbital_radius + planet_radius threshold for orbit
TOTAL_STEPS     = 500

# ─── Strategy constants ───────────────────────────────────────────────────────
MIN_KEEP         = 5         # always kept on any planet (hard floor)
MIN_FLEET        = 12        # minimum ships per fleet (speed floor)
BUFFER           = 3         # extra ships above predicted garrison
AIM_ITERS        = 8         # intercept-solver iterations
ETA_CAP          = 110       # ignore fleets / ships with ETA beyond this
PROD_RESERVE     = 9         # turns of production kept as reserve (normal)
PROD_RESERVE_LATE = 4        # same, late game
SUN_EXCL_R       = 22.0      # reject intercept destinations this close to sun
COMET_MIN_LIFE   = 18        # don't capture comets with fewer productive turns


# ═══════════════════════════════════════════════════════════════════════════════
#  PHYSICS
# ═══════════════════════════════════════════════════════════════════════════════

def pdist(ax, ay, bx, by):
    return math.hypot(ax - bx, ay - by)


def fleet_speed(ships):
    """Logarithmic speed curve — matches orbit_wars engine exactly."""
    if ships <= 1:
        return 1.0
    ratio = min(1.0, math.log(max(1, ships)) / math.log(1000.0))
    return 1.0 + (MAX_SPEED - 1.0) * (ratio ** 1.5)


def segment_hits_sun(x1, y1, x2, y2, safety=SUN_SAFETY):
    """True when segment (x1,y1)→(x2,y2) crosses the sun (with margin)."""
    r  = SUN_R + safety
    dx, dy = x2 - x1, y2 - y1
    fx, fy = x1 - CENTER_X, y1 - CENTER_Y
    a = dx * dx + dy * dy
    if a < 1e-9:
        return pdist(x1, y1, CENTER_X, CENTER_Y) < r
    b    = 2 * (fx * dx + fy * dy)
    c    = fx * fx + fy * fy - r * r
    disc = b * b - 4 * a * c
    if disc < 0:
        return False
    sq = math.sqrt(disc)
    t1 = (-b - sq) / (2 * a)
    t2 = (-b + sq) / (2 * a)
    return (0 <= t1 <= 1) or (0 <= t2 <= 1)


def safe_route(sx, sy, tx, ty):
    """
    Return (angle, effective_distance) for a sun-safe path src→dst.
    When the direct path is blocked, tries bypass waypoints on both sides
    of the sun using 7 clearance multipliers; picks the shortest clear detour.
    """
    if not segment_hits_sun(sx, sy, tx, ty):
        return math.atan2(ty - sy, tx - sx), pdist(sx, sy, tx, ty)

    vx, vy = tx - sx, ty - sy
    norm   = math.hypot(vx, vy)
    if norm < 1e-9:
        return math.atan2(ty - sy, tx - sx), pdist(sx, sy, tx, ty)

    nx, ny = -vy / norm, vx / norm      # perpendicular unit vector
    best   = None

    for sign in (1.0, -1.0):
        for mult in (1.2, 1.5, 1.9, 2.4, 3.0, 4.0, 5.5):
            wx = CENTER_X + sign * nx * SUN_R * mult
            wy = CENTER_Y + sign * ny * SUN_R * mult
            if segment_hits_sun(sx, sy, wx, wy) or segment_hits_sun(wx, wy, tx, ty):
                continue
            d = pdist(sx, sy, wx, wy) + pdist(wx, wy, tx, ty)
            if best is None or d < best[0]:
                best = (d, wx, wy)
            break       # tightest valid waypoint found for this side

    if best is None:
        # No clear bypass — inflate distance so this path scores poorly
        return math.atan2(ty - sy, tx - sx), pdist(sx, sy, tx, ty) * 2.5

    _, wx, wy = best
    return math.atan2(wy - sy, wx - sx), best[0]


def get_arrival(sx, sy, tx, ty, ships):
    """(angle, turns_int) for a fleet of `ships` travelling from (sx,sy) to (tx,ty)."""
    angle, d = safe_route(sx, sy, tx, ty)
    return angle, max(1, int(math.ceil(d / fleet_speed(ships))))


# ─── Planet-type predictors ───────────────────────────────────────────────────

def _orbiting_pos(planet, initial_by_id, ang_vel, turns):
    """Position after `turns` for any planet (static or orbiting)."""
    init = initial_by_id.get(planet.id)
    if init is None:
        return planet.x, planet.y
    r = pdist(init.x, init.y, CENTER_X, CENTER_Y)
    if r + init.radius >= ROTATION_LIMIT:
        return planet.x, planet.y          # static outer planet
    cur = math.atan2(planet.y - CENTER_Y, planet.x - CENTER_X)
    new = cur + ang_vel * turns
    return CENTER_X + r * math.cos(new), CENTER_Y + r * math.sin(new)


def _comet_pos(pid, comets, turns):
    """Position of comet planet `pid` after `turns`. Returns None if expired."""
    for g in comets:
        pids = g.get("planet_ids", [])
        if pid not in pids:
            continue
        idx  = pids.index(pid)
        paths = g.get("paths", [])
        pi   = g.get("path_index", 0)
        if idx >= len(paths):
            return None
        path = paths[idx]
        fi   = pi + int(turns)
        return (path[fi][0], path[fi][1]) if 0 <= fi < len(path) else None
    return None


def comet_life(pid, comets):
    """Turns remaining for comet planet `pid`."""
    for g in comets:
        pids = g.get("planet_ids", [])
        if pid not in pids:
            continue
        idx   = pids.index(pid)
        paths = g.get("paths", [])
        pi    = g.get("path_index", 0)
        return max(0, len(paths[idx]) - pi) if idx < len(paths) else 0
    return 0


def predict_pos(planet, initial_by_id, ang_vel, turns, comet_ids, comets):
    """Universal position prediction dispatching by planet type."""
    if planet.id in comet_ids:
        return _comet_pos(planet.id, comets, turns)    # may be None
    return _orbiting_pos(planet, initial_by_id, ang_vel, turns)


def aim_at(src, tgt, ships, initial_by_id, ang_vel, comets, comet_ids):
    """
    Iterative intercept solver.
    Returns (angle, turns, px, py) or None when target is unreachable or unsafe.

    Key safety checks:
      • Comet expired mid-convergence → None
      • Predicted intercept inside SUN_EXCL_R of sun → None  (catches fast
        comets swinging close to the sun — not worth chasing)
    """
    is_comet = tgt.id in comet_ids
    tol      = 0.4 if is_comet else 0.2

    tx, ty = tgt.x, tgt.y
    for _ in range(AIM_ITERS):
        angle, turns = get_arrival(src.x, src.y, tx, ty, ships)
        pos = predict_pos(tgt, initial_by_id, ang_vel, turns, comet_ids, comets)
        if pos is None:
            return None        # comet expired
        ntx, nty = pos

        # Reject intercepts in sun exclusion zone
        if pdist(ntx, nty, CENTER_X, CENTER_Y) < SUN_EXCL_R:
            return None        # sun-proximal comet perihelion — skip it

        if abs(ntx - tx) < tol and abs(nty - ty) < tol:
            tx, ty = ntx, nty
            break
        tx, ty = ntx, nty

    angle, turns = get_arrival(src.x, src.y, tx, ty, ships)
    return angle, turns, tx, ty


# ═══════════════════════════════════════════════════════════════════════════════
#  FLEET TRACKING  (geometric projection — not angle comparison)
# ═══════════════════════════════════════════════════════════════════════════════

def fleet_target(f, planets):
    """Which planet is fleet f heading toward?  Returns (planet, eta) or (None, None)."""
    fvx, fvy = math.cos(f.angle), math.sin(f.angle)
    sp = fleet_speed(f.ships)
    best_p, best_t = None, 1e9
    for p in planets:
        dx, dy = p.x - f.x, p.y - f.y
        proj   = dx * fvx + dy * fvy
        if proj <= 0:
            continue
        perp = abs(dx * fvy - dy * fvx)
        if perp > p.radius + 1.2:
            continue
        t = proj / sp
        if t < best_t and t <= ETA_CAP:
            best_t, best_p = t, p
    return (best_p, int(math.ceil(best_t))) if best_p else (None, None)


def fleets_to(planet, fleets):
    """All (eta, owner, ships) tuples for fleets heading to planet."""
    arrivals = []
    for f in fleets:
        fvx, fvy = math.cos(f.angle), math.sin(f.angle)
        dx, dy   = planet.x - f.x, planet.y - f.y
        proj     = dx * fvx + dy * fvy
        if proj <= 0:
            continue
        perp = abs(dx * fvy - dy * fvx)
        if perp > planet.radius + 1.5:
            continue
        t = proj / fleet_speed(f.ships)
        if t > ETA_CAP:
            continue
        arrivals.append((int(math.ceil(t)), f.owner, int(f.ships)))
    return arrivals


# ═══════════════════════════════════════════════════════════════════════════════
#  BATTLE SIMULATION
# ═══════════════════════════════════════════════════════════════════════════════

def defense_shortfall(planet, arrivals, player):
    """
    Event-driven garrison simulation.
    Returns extra ships needed right now so this planet survives all waves.
    0 = safe.
    """
    if not arrivals:
        return 0
    events   = sorted(arrivals)
    garrison = planet.ships
    prod     = planet.production if planet.owner == player else 0
    last_t   = 0
    deficit  = 0
    i = 0
    while i < len(events):
        t = events[i][0]
        garrison += (t - last_t) * prod
        group = []
        while i < len(events) and events[i][0] == t:
            group.append(events[i])
            i += 1
        friendly = sum(s for _, o, s in group if o == player)
        enemy    = sum(s for _, o, s in group if o != player)
        garrison += friendly - enemy
        if garrison < 0:
            deficit = max(deficit, -garrison)
        last_t = t
    return deficit


def garrison_at_arrival(tgt, eta_turns, early_enemy):
    """
    Garrison state of `tgt` when OUR fleet arrives at eta_turns,
    after processing enemy fleets that arrive BEFORE us.
    Returns (estimated_garrison, owner_then).
    """
    if not early_enemy:
        if tgt.owner == -1:
            return tgt.ships, -1
        return tgt.ships + tgt.production * eta_turns, tgt.owner

    events   = sorted(early_enemy)          # [(eta, ships), ...]
    garrison = tgt.ships
    owner    = tgt.owner
    prod     = tgt.production
    last_t   = 0

    for t, o_ships in events:
        if owner != -1:
            garrison += (t - last_t) * prod
        if o_ships > garrison:
            owner    = -2   # captured by unknown enemy
            garrison = o_ships - garrison
        else:
            garrison -= o_ships
        last_t = t

    if owner != -1:
        garrison += (eta_turns - last_t) * prod
    return max(0, int(garrison)), owner


# ═══════════════════════════════════════════════════════════════════════════════
#  MAIN AGENT
# ═══════════════════════════════════════════════════════════════════════════════

def agent(obs):

    # ── Phase 0: Parse & classify ────────────────────────────────────────────
    get = obs.get if isinstance(obs, dict) else lambda k, d=None: getattr(obs, k, d)

    player    = get("player", 0)
    step      = get("step", 0) or 0
    ang_vel   = get("angular_velocity", 0.0) or 0.0
    raw_pl    = get("planets", []) or []
    raw_fl    = get("fleets",  []) or []
    raw_init  = get("initial_planets", []) or []
    comets    = get("comets", []) or []
    comet_ids = set(get("comet_planet_ids", []) or [])

    planets       = [Planet(*p) for p in raw_pl]
    fleets        = [Fleet(*f)  for f in raw_fl]
    initial_by_id = {Planet(*p).id: Planet(*p) for p in raw_init}

    my_planets    = [p for p in planets if p.owner == player]
    enemy_planets = [p for p in planets if p.owner not in (player, -1)]
    other_planets = [p for p in planets if p.owner != player]

    if not my_planets:
        return []

    # ── Game-state flags ─────────────────────────────────────────────────────
    remaining    = max(1, TOTAL_STEPS - step)
    is_early     = step < 70
    is_late      = remaining < 80
    is_very_late = remaining < 35

    # Count active players (adapts strategy)
    active_owners  = {p.owner for p in planets if p.owner >= 0}
    active_owners |= {f.owner for f in fleets  if f.owner >= 0}
    multiplayer    = len(active_owners) >= 3

    # Ship totals per player
    ships_by = {}
    for p in planets:
        if p.owner >= 0:
            ships_by[p.owner] = ships_by.get(p.owner, 0) + p.ships
    for f in fleets:
        if f.owner >= 0:
            ships_by[f.owner] = ships_by.get(f.owner, 0) + f.ships

    my_ships  = ships_by.get(player, 0)
    enemy_max = max((v for k, v in ships_by.items() if k != player), default=1)
    trailing  = my_ships < enemy_max * 0.80
    winning   = my_ships > enemy_max * 1.35

    # ── Planet type helper ───────────────────────────────────────────────────
    def is_rotating(p):
        init = initial_by_id.get(p.id)
        if init is None:
            return False
        return pdist(init.x, init.y, CENTER_X, CENTER_Y) + init.radius < ROTATION_LIMIT

    # ── Fleet tracking ───────────────────────────────────────────────────────
    # committed_to[tid] = friendly ships currently in-flight to planet tid
    committed_to  = {}
    inbound_enemy = {}   # tid → [(eta, ships), ...]

    for f in fleets:
        tgt, eta = fleet_target(f, planets)
        if tgt is None:
            continue
        if f.owner == player:
            committed_to[tgt.id] = committed_to.get(tgt.id, 0) + f.ships
        else:
            inbound_enemy.setdefault(tgt.id, []).append((eta, f.ships))

    # ── Counter-attack: freshly weakened enemy planets ───────────────────────
    # from_planet_id reveals which planet each enemy fleet came from
    enemy_shipped_from = {}
    for f in fleets:
        if f.owner != player:
            fpid = getattr(f, "from_planet_id", -1)
            if fpid >= 0:
                enemy_shipped_from[fpid] = (enemy_shipped_from.get(fpid, 0)
                                            + f.ships)

    # ── Strategic-enemy analysis ─────────────────────────────────────────────
    # Identify each enemy owner's maximum production planet
    enemy_max_prod = {}
    for p in enemy_planets:
        enemy_max_prod[p.owner] = max(enemy_max_prod.get(p.owner, 0),
                                      p.production)

    def is_strategic(tgt):
        return (tgt.owner not in (player, -1)
                and tgt.production >= enemy_max_prod.get(tgt.owner, 0))

    # ── Race-condition analysis ───────────────────────────────────────────────
    def enemy_min_eta(tgt):
        """Rough minimum ETA for any enemy to reach tgt."""
        m = float("inf")
        for eta_t, _ in inbound_enemy.get(tgt.id, []):
            m = min(m, eta_t)
        for ep in enemy_planets:
            d = pdist(ep.x, ep.y, tgt.x, tgt.y)
            m = min(m, d / fleet_speed(30))   # 30 ships ≈ decent speed proxy
        return m

    # ── Defense reserves ─────────────────────────────────────────────────────
    prod_res = PROD_RESERVE_LATE if is_late else PROD_RESERVE

    reserve = {}
    for p in my_planets:
        arrivals = fleets_to(p, fleets)
        need     = defense_shortfall(p, arrivals, player)
        prod_buf = min(p.ships // 3, p.production * prod_res)
        reserve[p.id] = min(p.ships - MIN_KEEP, max(0, need) + prod_buf)

    available = {p.id: max(0, p.ships - max(MIN_KEEP, reserve[p.id]))
                 for p in my_planets}

    moves = []

    # ── Local helpers ────────────────────────────────────────────────────────
    def depart_ok(src, angle):
        """Reject angles that immediately point toward the sun."""
        return not segment_hits_sun(
            src.x, src.y,
            src.x + math.cos(angle) * 5,
            src.y + math.sin(angle) * 5,
            safety=0.5
        )

    def compute_needed(tgt, eta_turns):
        """
        Ships required to capture tgt, accounting for:
          • garrison growth  (production during transit)
          • enemy fleets that arrive BEFORE ours  (may flip ownership)
          • already-committed friendly ships
        """
        early_en = [(t, s) for t, s in inbound_enemy.get(tgt.id, [])
                    if t <= eta_turns]
        if early_en:
            g, _ = garrison_at_arrival(tgt, eta_turns, early_en)
        elif tgt.owner == -1:
            g = tgt.ships                                   # neutrals don't grow
        else:
            g = tgt.ships + tgt.production * eta_turns     # enemy grows

        g -= committed_to.get(tgt.id, 0)
        return max(0, int(g)) + BUFFER + 1

    def score_target(tgt, eta_turns, needed):
        """
        Composite ROI score (higher = better).
        Core formula:
            (production × productive_turns × multipliers)
            ─────────────────────────────────────────────
            needed × (1 + eta × 0.018)
        """
        if needed <= 0:
            return 0.0

        prod_turns = max(0, remaining - eta_turns)

        if tgt.id in comet_ids:
            life = comet_life(tgt.id, comets)
            prod_turns = min(prod_turns, max(0, life - eta_turns))
            if prod_turns < COMET_MIN_LIFE:
                return -1.0     # not worth the effort

        base = tgt.production * prod_turns
        if base <= 0 and not is_very_late:
            return -1.0

        mult = 1.0

        # ── Enemy planet ──────────────────────────────────────────────────
        if tgt.owner not in (player, -1):
            mult *= 1.80        # we gain + they lose (denial)

            if is_strategic(tgt):
                mult *= 1.25    # their most important planet

            # Freshly weakened: they just sent ships out
            shipped = enemy_shipped_from.get(tgt.id, 0)
            if shipped > 0:
                weakness = min(0.65, shipped / max(1, tgt.ships + shipped))
                mult *= 1.0 + weakness * 0.9

        # ── Race condition on neutral ─────────────────────────────────────
        if tgt.owner == -1:
            en_eta = enemy_min_eta(tgt)
            if en_eta < float("inf"):
                ratio = en_eta / max(1.0, float(eta_turns))
                if ratio < 1.35:
                    mult *= 1.45    # enemy is close behind — urgent!

        # ── Phase multipliers ─────────────────────────────────────────────
        if is_early and tgt.owner == -1 and tgt.id not in comet_ids:
            mult *= 1.50        # rush static/orbiting neutrals before enemies do

        if trailing:
            if tgt.owner not in (player, -1):
                mult *= 1.55    # must reduce enemy production when behind
            else:
                mult *= 1.15

        # ── 4-player adaptation ───────────────────────────────────────────
        if multiplayer:
            if tgt.owner == -1:
                mult *= 1.20    # let enemies fight; prefer free neutrals
            elif tgt.owner not in (player, -1):
                tgt_str = ships_by.get(tgt.owner, 0)
                min_en  = min((v for k, v in ships_by.items()
                               if k != player), default=1)
                if tgt_str <= min_en * 1.10:
                    mult *= 1.15   # concentrate on weakest enemy
                else:
                    mult *= 0.65   # avoid two-front war against stronger player

        # ── Orbiting bonus (harder for enemies to aim at) ─────────────────
        if is_rotating(tgt) and tgt.id not in comet_ids:
            mult *= 1.05

        return (base * mult) / (needed * (1.0 + eta_turns * 0.018))

    def dispatch(src_id, angle, send, tgt_id):
        moves.append([src_id, float(angle), int(send)])
        available[src_id]    -= send
        committed_to[tgt_id]  = committed_to.get(tgt_id, 0) + send

    # ════════════════════════════════════════════════════════════════════════
    #  Phase 1 — Emergency reinforcement
    # ════════════════════════════════════════════════════════════════════════
    threatened = []
    for p in my_planets:
        arrivals = fleets_to(p, fleets)
        need     = defense_shortfall(p, arrivals, player)
        already  = committed_to.get(p.id, 0)
        deficit  = need - already
        if deficit > 0:
            roi = p.production * remaining - deficit
            if roi > 0 or p.ships >= 15:
                threatened.append((deficit, p))

    threatened.sort(key=lambda x: -x[0])

    for deficit, at_risk in threatened:
        still = deficit + BUFFER
        helpers = sorted(
            [q for q in my_planets
             if q.id != at_risk.id
             and available[q.id] >= MIN_FLEET
             and pdist(q.x, q.y, at_risk.x, at_risk.y) <= 65],
            key=lambda q: pdist(q.x, q.y, at_risk.x, at_risk.y)
        )
        for helper in helpers:
            if still <= 0:
                break
            send = min(available[helper.id], still)
            if send < MIN_FLEET:
                continue
            res = aim_at(helper, at_risk, send,
                         initial_by_id, ang_vel, comets, comet_ids)
            if res is None:
                continue
            angle, turns, _, _ = res
            if not depart_ok(helper, angle):
                continue
            dispatch(helper.id, angle, send, at_risk.id)
            still -= send

    # ════════════════════════════════════════════════════════════════════════
    #  Phase 2 — Target-first multi-source coordinated attacks
    # ════════════════════════════════════════════════════════════════════════

    # Score every attackable planet
    scored_targets = []
    for tgt in other_planets:
        if tgt.id in comet_ids and comet_life(tgt.id, comets) < COMET_MIN_LIFE:
            continue

        # Use closest available source for rough ETA
        best_src = min(
            (p for p in my_planets if available[p.id] >= MIN_FLEET),
            key=lambda p: pdist(p.x, p.y, tgt.x, tgt.y),
            default=None
        )
        if best_src is None:
            continue

        res = aim_at(best_src, tgt, max(MIN_FLEET, available[best_src.id]),
                     initial_by_id, ang_vel, comets, comet_ids)
        if res is None:
            continue
        _, rough_eta, _, _ = res

        if is_late and rough_eta > remaining - 3:
            continue

        need = compute_needed(tgt, rough_eta)
        if need <= 0:
            continue

        sc = score_target(tgt, rough_eta, need)
        if sc <= 0 and not is_very_late:
            continue

        scored_targets.append((sc, tgt))

    scored_targets.sort(key=lambda x: -x[0])

    # Greedy multi-source assignment (target-first order)
    for _sc, tgt in scored_targets:
        sources = sorted(
            [p for p in my_planets if available[p.id] >= MIN_FLEET],
            key=lambda p: pdist(p.x, p.y, tgt.x, tgt.y)
        )
        for src in sources:
            avail = available[src.id]
            if avail < MIN_FLEET:
                continue

            res = aim_at(src, tgt, avail,
                         initial_by_id, ang_vel, comets, comet_ids)
            if res is None:
                continue
            angle, turns, _, _ = res

            if not depart_ok(src, angle):
                continue
            if is_late and turns > remaining - 2:
                continue
            if tgt.id in comet_ids and turns >= comet_life(tgt.id, comets):
                continue

            exact_need = compute_needed(tgt, turns)
            if exact_need <= 0:
                break       # fully covered — stop recruiting more sources

            send = max(MIN_FLEET, min(avail, exact_need))
            if send > avail:
                continue

            dispatch(src.id, angle, send, tgt.id)

    # ════════════════════════════════════════════════════════════════════════
    #  Phase 3 — Secondary pass: leftover ships chip in on partial coverage
    # ════════════════════════════════════════════════════════════════════════
    if not is_very_late:
        for src in my_planets:
            avail = available[src.id]
            if avail < MIN_FLEET * 2:
                continue

            best_sc, best_tgt, best_angle, best_send = -1.0, None, None, 0

            for tgt in other_planets:
                if tgt.id in comet_ids and comet_life(tgt.id, comets) < COMET_MIN_LIFE:
                    continue

                res = aim_at(src, tgt, avail,
                             initial_by_id, ang_vel, comets, comet_ids)
                if res is None:
                    continue
                angle, turns, _, _ = res

                if not depart_ok(src, angle):
                    continue
                if is_late and turns > remaining - 2:
                    continue
                if tgt.id in comet_ids and turns >= comet_life(tgt.id, comets):
                    continue

                need = compute_needed(tgt, turns)
                if need <= 0 or need > avail:
                    continue

                sc = score_target(tgt, turns, need)
                if sc > best_sc:
                    best_sc, best_tgt  = sc, tgt
                    best_angle, best_send = angle, need

            if best_tgt is not None and best_sc > 0:
                send = max(MIN_FLEET, min(available[src.id], best_send))
                if send <= available[src.id]:
                    dispatch(src.id, best_angle, send, best_tgt.id)

    # ════════════════════════════════════════════════════════════════════════
    #  Phase 4 — Frontline consolidation
    # ════════════════════════════════════════════════════════════════════════
    if my_planets and enemy_planets and not is_very_late:
        front_d = {mp.id: min(pdist(mp.x, mp.y, e.x, e.y)
                              for e in enemy_planets)
                   for mp in my_planets}
        front   = min(my_planets, key=lambda p: front_d[p.id])

        for rear in my_planets:
            if rear.id == front.id:
                continue
            if front_d[rear.id] < front_d[front.id] * 1.4:
                continue        # not far enough in the rear
            avail = available[rear.id]
            send  = int(avail * (0.78 if is_late else 0.62))
            if send < MIN_FLEET:
                continue
            res = aim_at(rear, front, send,
                         initial_by_id, ang_vel, comets, comet_ids)
            if res is None:
                continue
            angle, turns, _, _ = res
            if turns > 50 or not depart_ok(rear, angle):
                continue
            dispatch(rear.id, angle, send, front.id)

    # ════════════════════════════════════════════════════════════════════════
    #  Phase 5 — Endgame all-in  (last ~35 turns)
    # ════════════════════════════════════════════════════════════════════════
    if is_very_late and other_planets:
        for src in my_planets:
            avail = available[src.id]
            if avail < MIN_FLEET:
                continue

            best = None
            for tgt in other_planets:
                res = aim_at(src, tgt, avail,
                             initial_by_id, ang_vel, comets, comet_ids)
                if res is None:
                    continue
                angle, turns, _, _ = res
                if turns > remaining or not depart_ok(src, angle):
                    continue
                sc = (tgt.production * max(0, remaining - turns)
                      - pdist(src.x, src.y, tgt.x, tgt.y) * 0.3)
                if best is None or sc > best[0]:
                    best = (sc, angle, tgt.id)

            if best is None:
                continue
            _, angle, tid = best
            dispatch(src.id, angle, avail, tid)

    return moves

Writing submission.py
